# Results for model: baichuan-inc/baichuan2-13b-chat

In [ ]:
import polars as pl
import xgboost as xgb
import pandas as pd
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

# Load the train.parquet file using Polars
data = pl.read_parquet('./jane_street_data/train.parquet')

# Feature Engineering: Calculate a global TOP_QUANTILE (top 15%) of 'feature_00'
# relative to 'responder_6' across rolling batches of 'date_id'.
def top_quantile(data, feature, responder, window_size=100):
    return data.where(f'{feature}_00', window=window_size) \
                 .groupby('date_id') \
                 .orderBy(f'{feature}_00', ascending=False) \
                 .window(window_size, drop=True) \
                 . sign('feature_00') \
                 .quantile(0.85) \
                 .alias(f'{feature}_TOP_QUANTILE')

# Calculate top quantile for each rolling batch
data = data.with_column(top_quantile('feature_00', 'date_id', 'responder_6'))

# Split data into train and test sets
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

# Train an XGBoost classifier on the target 'responder_6'
X_train = train_data.select(['feature_00', 'date_id', 'responder_6', 'responder_6_TOP_QUANTILE'])
y_train = train_data['responder_6']
X_test = test_data.select(['feature_00', 'date_id', 'responder_6', 'responder_6_TOP_QUANTILE'])

model = xgb.XGBRegressor()
model.fit(X_train, y_train, eval_metric='rmse')

# Make predictions and calculate mean squared error
y_pred = model.predict(X_test)
mse = mean_squared_error(y_train, y_pred)
print(f"Mean Squared Error: {mse}")

# Save the model for future use
model.save_model('jane_street_market_forecasting_model.bin')